In [ ]:
#|hide
#| eval: false
! [ -e /content ] && pip install -Uqq fastai  # upgrade fastai on colab

In [ ]:
#|default_exp callback.swa

In [ ]:
#|export
from __future__ import annotations
import copy
from fastai.basics import *

In [ ]:
#|hide
from nbdev.showdoc import *
from fastai.test_utils import *

# Stochastic Weight Averaging

> A callback that maintains an exponential moving average of model weights and swaps to the averaged weights at evaluation time for improved generalization.

In [ ]:
#|export
class StochasticWeightAveraging(Callback):
    "A `Callback` that maintains an exponential moving average of model weights and swaps to the averaged weights at evaluation time."
    order = 65  # After TrackerCallback (60) but before most other callbacks

    def __init__(self,
        decay:float=0.999,  # EMA decay factor; higher means slower update of the average.
        start_epoch:int=0,  # Epoch at which to start accumulating the EMA.
        save_averaged:bool=False,  # If True, save the averaged model at end of training.
        fname:str='swa_model'  # Filename used when saving the averaged model.
    ):
        assert 0. < decay < 1., f"decay must be between 0 and 1, got {decay}"
        assert start_epoch >= 0, f"start_epoch must be non-negative, got {start_epoch}"
        store_attr('decay,start_epoch,save_averaged,fname')

    def before_fit(self):
        "Initialize the EMA shadow weights as a copy of the model state_dict."
        self.run = not hasattr(self, "lr_finder") and not hasattr(self, "gather_preds")
        if not self.run: return
        self.ema_state = copy.deepcopy(self.model.state_dict())
        self._training_state = None

    def after_epoch(self):
        "Update the EMA weights after each training epoch (once past start_epoch)."
        if self.epoch < self.start_epoch: return
        model_state = self.model.state_dict()
        for key in self.ema_state:
            if self.ema_state[key].is_floating_point():
                self.ema_state[key].mul_(self.decay).add_(model_state[key], alpha=1.0 - self.decay)
            else:
                self.ema_state[key].copy_(model_state[key])

    def before_validate(self):
        "Swap model weights to EMA weights before validation."
        if not hasattr(self, 'ema_state'): return
        self._training_state = copy.deepcopy(self.model.state_dict())
        self.model.load_state_dict(self.ema_state)

    def after_validate(self):
        "Restore training weights after validation."
        if self._training_state is not None:
            self.model.load_state_dict(self._training_state)
            self._training_state = None

    def after_fit(self):
        "Load EMA weights into the model at the end of training, optionally saving them."
        if hasattr(self, 'ema_state'):
            self.model.load_state_dict(self.ema_state)
            if self.save_averaged:
                self.learn.save(self.fname)

## Usage

`StochasticWeightAveraging` keeps a running exponential moving average (EMA) of model weights throughout training. Before each validation pass it swaps the averaged weights into the model so that validation metrics reflect the smoothed parameters. After validation, the training weights are restored so gradient updates continue normally.

At the end of training the model is left with the EMA weights loaded, giving you a model that typically generalizes better than the final iterate.

In [ ]:
learn = synth_learner(n_trn=5)
learn.fit(3, cbs=StochasticWeightAveraging(decay=0.9))

In [ ]:
#|hide
# Test that the callback runs without error and produces a model
assert learn.model is not None

## Export -

In [ ]:
#|hide
from nbdev import nbdev_export
nbdev_export()